In [ ]:
# ============================================================
# ARABIC PDF OCR
# CPU / Windows version
# ============================================================

# IMPORTANT:
# Disable PaddlePaddle oneDNN before importing PaddleOCR
import os

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"


# ============================================================
# IMPORTS
# ============================================================

import json
import pymupdf
from PIL import Image
from paddleocr import PaddleOCR


# ============================================================
# 1. PDF PATH
# ============================================================

PDF_PATH = r"C:\Users\PC-LAB1\Desktop\llm\الجريدة-الرسمية-دولة-ليبيا-ديوان-مجلس-النواب-العدد-الثاني.pdf"


# ============================================================
# 2. OUTPUT FOLDER
# ============================================================

PDF_DIR = os.path.dirname(PDF_PATH)

OUTPUT_DIR = os.path.join(
    PDF_DIR,
    "ocr_output"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

TEXT_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "arabic_ocr.txt"
)

JSON_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "arabic_ocr.json"
)


# ============================================================
# 3. CHECK PDF
# ============================================================

if not os.path.isfile(PDF_PATH):

    raise FileNotFoundError(
        f"\nPDF not found:\n{PDF_PATH}"
    )

print("PDF found:")
print(PDF_PATH)


# ============================================================
# 4. LOAD OCR
# ============================================================

print("\nLoading Arabic OCR model...")

ocr = PaddleOCR(
    lang="ar"
)

print("Arabic OCR model loaded.")


# ============================================================
# 5. OPEN PDF
# ============================================================

pdf = pymupdf.open(
    PDF_PATH
)

total_pages = len(pdf)

print(
    f"\nTotal pages: {total_pages}"
)


# ============================================================
# 6. PROCESS PAGES
# ============================================================

all_pages = []


for page_number in range(total_pages):

    print("\n" + "=" * 70)

    print(
        f"Processing page "
        f"{page_number + 1} / {total_pages}"
    )

    print("=" * 70)


    # ========================================================
    # GET PAGE
    # ========================================================

    page = pdf[page_number]


    # ========================================================
    # RENDER PDF PAGE
    # ========================================================

    zoom = 3.0

    matrix = pymupdf.Matrix(
        zoom,
        zoom
    )

    pix = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )


    # ========================================================
    # CONVERT TO PIL IMAGE
    # ========================================================

    image = Image.frombytes(
        "RGB",
        (
            pix.width,
            pix.height
        ),
        pix.samples
    )


    # ========================================================
    # SAVE PAGE IMAGE
    # ========================================================

    image_path = os.path.join(
        OUTPUT_DIR,
        f"page_{page_number + 1:03d}.png"
    )

    image.save(
        image_path
    )


    # ========================================================
    # OCR
    # ========================================================

    result = ocr.predict(
        image_path
    )


    # ========================================================
    # EXTRACT TEXT
    # ========================================================

    page_text = []


    for res in result:

        try:

            data = res.json

        except Exception:

            continue


        if isinstance(
            data,
            str
        ):

            try:

                data = json.loads(
                    data
                )

            except Exception:

                continue


        if not isinstance(
            data,
            dict
        ):

            continue


        texts = data.get(
            "rec_texts",
            []
        )

        scores = data.get(
            "rec_scores",
            []
        )


        for i, text in enumerate(texts):

            if text is None:

                continue


            text = str(
                text
            ).strip()


            if not text:

                continue


            if i < len(scores):

                try:

                    confidence = float(
                        scores[i]
                    )

                except Exception:

                    confidence = 0.0

            else:

                confidence = 0.0


            if confidence >= 0.30:

                page_text.append(
                    {
                        "text": text,
                        "confidence": confidence
                    }
                )


    # ========================================================
    # SAVE PAGE
    # ========================================================

    all_pages.append(
        {
            "page": page_number + 1,
            "text": page_text
        }
    )


    print(
        f"Detected "
        f"{len(page_text)} text elements."
    )


# ============================================================
# 7. CLOSE PDF
# ============================================================

pdf.close()


# ============================================================
# 8. SAVE TXT
# ============================================================

with open(
    TEXT_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    for page in all_pages:

        f.write(
            "\n"
            + "=" * 80
            + "\n"
        )

        f.write(
            f"الصفحة {page['page']}\n"
        )

        f.write(
            "=" * 80
            + "\n\n"
        )


        for item in page["text"]:

            f.write(
                item["text"]
                + "\n"
            )


# ============================================================
# 9. SAVE JSON
# ============================================================

with open(
    JSON_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_pages,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n")

print("=" * 80)

print("OCR COMPLETE")

print("=" * 80)


for page in all_pages:

    print(
        f"\n\n"
        f"========== الصفحة {page['page']} ==========\n"
    )


    for item in page["text"]:

        print(
            item["text"]
        )


# ============================================================
# 11. OUTPUT FILES
# ============================================================

print("\n")

print("=" * 80)

print("FILES SAVED")

print("=" * 80)

print(
    "\nTXT:"
)

print(
    TEXT_OUTPUT
)

print(
    "\nJSON:"
)

print(
    JSON_OUTPUT
)

print(
    "\nPage images:"
)

print(
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# ARABIC PDF OCR - ONE PAGE TEST
# ============================================================

import os
import json
import torch
import pymupdf

from paddleocr import PaddleOCR


# ============================================================
# 1. PDF
# ============================================================

PDF_PATH = r"C:\Users\PC-LAB1\Desktop\llm\الجريدة-الرسمية-دولة-ليبيا-ديوان-مجلس-النواب-العدد-الثاني.pdf"


# ============================================================
# 2. CHECK PDF
# ============================================================

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(PDF_PATH)

print("PDF found.")


# ============================================================
# 3. LOAD OCR
# ============================================================

print("Loading Arabic OCR...")

ocr = PaddleOCR(
    lang="ar"
)

print("OCR loaded successfully.")


# ============================================================
# 4. OPEN PDF
# ============================================================

pdf = pymupdf.open(PDF_PATH)

print("Total pages:", len(pdf))


# ============================================================
# 5. RENDER FIRST PAGE
# ============================================================

page = pdf[0]

zoom = 3.0

matrix = pymupdf.Matrix(
    zoom,
    zoom
)

pix = page.get_pixmap(
    matrix=matrix,
    alpha=False
)


# ============================================================
# 6. SAVE TEST IMAGE
# ============================================================

output_dir = os.path.join(
    os.path.dirname(PDF_PATH),
    "ocr_output"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

image_path = os.path.join(
    output_dir,
    "test_page_001.png"
)

pix.save(image_path)

print("Page image saved:")
print(image_path)


# ============================================================
# 7. RUN OCR
# ============================================================

print("\nRunning OCR on page 1...")

result = ocr.predict(
    image_path
)

print("OCR finished!")


# ============================================================
# 8. EXTRACT RESULTS
# ============================================================

all_text = []

for res in result:

    data = res.json

    if isinstance(data, str):
        data = json.loads(data)

    texts = data.get(
        "rec_texts",
        []
    )

    scores = data.get(
        "rec_scores",
        []
    )

    for i, text in enumerate(texts):

        if not text:
            continue

        text = str(text).strip()

        if i < len(scores):
            score = float(scores[i])
        else:
            score = 0.0

        all_text.append(
            {
                "text": text,
                "confidence": score
            }
        )


# ============================================================
# 9. DISPLAY
# ============================================================

print("\n")
print("=" * 80)
print("ARABIC OCR RESULT")
print("=" * 80)

for item in all_text:

    print(
        f"{item['confidence']:.3f} | "
        f"{item['text']}"
    )


print("\nDetected text elements:", len(all_text))

pdf.close()

In [ ]:
# ============================================================
# ARABIC PDF OCR - PADDLEOCR
# Windows / VS Code / Jupyter
# ============================================================

import os
import json
import time
import numpy as np

# ------------------------------------------------------------
# IMPORTANT: Set Paddle flags BEFORE importing Paddle
# ------------------------------------------------------------
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

# Import torch FIRST to avoid Windows DLL conflict
import torch

import pymupdf
from paddleocr import PaddleOCR


# ============================================================
# PATHS
# ============================================================

PDF_PATH = r"C:\Users\PC-LAB1\Desktop\llm\الجريدة-الرسمية-دولة-ليبيا-ديوان-مجلس-النواب-العدد-الثاني.pdf"

OUTPUT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output"

TXT_PATH = os.path.join(OUTPUT_DIR, "arabic_ocr.txt")
JSON_PATH = os.path.join(OUTPUT_DIR, "arabic_ocr.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CHECK FILE
# ============================================================

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found:\n{PDF_PATH}"
    )

print("=" * 80)
print("ARABIC PDF OCR")
print("=" * 80)

print(f"\nPDF:")
print(PDF_PATH)

print(f"\nOutput folder:")
print(OUTPUT_DIR)


# ============================================================
# OPEN PDF
# ============================================================

pdf = pymupdf.open(PDF_PATH)

print(f"\nNumber of pages: {len(pdf)}")


# ============================================================
# LOAD PADDLEOCR
# ============================================================

print("\nLoading PaddleOCR...")

ocr = PaddleOCR(
    lang="ar"
)

print("PaddleOCR loaded successfully.")


# ============================================================
# OCR SETTINGS
# ============================================================

# Lower detection threshold helps with scanned/document text
TEXT_DET_THRESH = 0.20
TEXT_DET_BOX_THRESH = 0.30

# Allow recognition results even with relatively low confidence
TEXT_REC_SCORE_THRESH = 0.0

# ============================================================
# STORAGE
# ============================================================

all_pages = []

total_text_lines = 0

start_time = time.time()


# ============================================================
# PROCESS EVERY PAGE
# ============================================================

for page_number in range(len(pdf)):

    page_start = time.time()

    print("\n" + "=" * 80)
    print(f"PAGE {page_number + 1} / {len(pdf)}")
    print("=" * 80)

    page = pdf[page_number]

    # --------------------------------------------------------
    # Render PDF page directly into memory
    # No PNG files are created
    # --------------------------------------------------------

    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(2.0, 2.0),
        alpha=False
    )

    # PyMuPDF returns RGB bytes
    image = np.frombuffer(
        pix.samples,
        dtype=np.uint8
    )

    image = image.reshape(
        pix.height,
        pix.width,
        pix.n
    )

    # Make sure image has exactly 3 channels
    if image.shape[2] == 4:
        image = image[:, :, :3]

    # --------------------------------------------------------
    # RUN OCR
    # --------------------------------------------------------

    try:

        result = ocr.predict(
            image,

            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,

            text_det_thresh=TEXT_DET_THRESH,
            text_det_box_thresh=TEXT_DET_BOX_THRESH,

            text_rec_score_thresh=TEXT_REC_SCORE_THRESH
        )

    except Exception as e:

        print("\nOCR ERROR:")
        print(str(e))

        all_pages.append({
            "page": page_number + 1,
            "text": "",
            "elements": [],
            "error": str(e)
        })

        continue


    # ========================================================
    # EXTRACT OCR RESULT
    # ========================================================

    page_elements = []

    page_text = []


    # PaddleOCR 3.x returns a list of result objects
    for res in result:

        # ----------------------------------------------------
        # Get dictionary
        # ----------------------------------------------------

        try:
            data = res.json
        except Exception:
            data = None

        # ----------------------------------------------------
        # Convert JSON string if necessary
        # ----------------------------------------------------

        if isinstance(data, str):

            try:
                data = json.loads(data)
            except Exception:
                data = None


        if not isinstance(data, dict):
            continue


        # ----------------------------------------------------
        # IMPORTANT:
        # PaddleOCR 3.x stores OCR information inside "res"
        # ----------------------------------------------------

        if "res" in data and isinstance(data["res"], dict):
            data = data["res"]


        # ----------------------------------------------------
        # Get OCR fields
        # ----------------------------------------------------

        dt_polys = data.get("dt_polys", [])
        dt_scores = data.get("dt_scores", [])

        rec_texts = data.get("rec_texts", [])
        rec_scores = data.get("rec_scores", [])


        # Convert to lists safely
        try:
            dt_polys = list(dt_polys)
        except Exception:
            dt_polys = []

        try:
            dt_scores = list(dt_scores)
        except Exception:
            dt_scores = []

        try:
            rec_texts = list(rec_texts)
        except Exception:
            rec_texts = []

        try:
            rec_scores = list(rec_scores)
        except Exception:
            rec_scores = []


        print(
            f"Detected boxes: {len(dt_polys)} | "
            f"Recognized texts: {len(rec_texts)}"
        )


        # ----------------------------------------------------
        # Extract every recognized text
        # ----------------------------------------------------

        for i, text in enumerate(rec_texts):

            if text is None:
                continue

            text = str(text).strip()

            if not text:
                continue


            # Recognition confidence
            confidence = 0.0

            if i < len(rec_scores):

                try:
                    confidence = float(rec_scores[i])
                except Exception:
                    confidence = 0.0


            # Detection confidence
            detection_confidence = 0.0

            if i < len(dt_scores):

                try:
                    detection_confidence = float(dt_scores[i])
                except Exception:
                    detection_confidence = 0.0


            # Bounding box
            box = None

            if i < len(dt_polys):

                try:
                    box = np.asarray(
                        dt_polys[i]
                    ).tolist()
                except Exception:
                    box = None


            item = {
                "text": text,
                "confidence": confidence,
                "detection_confidence": detection_confidence,
                "box": box
            }

            page_elements.append(item)
            page_text.append(text)


    # ========================================================
    # SAVE PAGE RESULT IN MEMORY
    # ========================================================

    page_text_string = "\n".join(page_text)

    page_result = {
        "page": page_number + 1,
        "text": page_text_string,
        "elements": page_elements
    }

    all_pages.append(page_result)

    total_text_lines += len(page_elements)


    # --------------------------------------------------------
    # Display page result
    # --------------------------------------------------------

    print(
        f"\nText elements found on page: "
        f"{len(page_elements)}"
    )

    if page_elements:

        print("\nRecognized text:")

        for item in page_elements:

            print(
                f"{item['confidence']:.3f} | "
                f"{item['text']}"
            )

    else:

        print("\nNO TEXT DETECTED ON THIS PAGE.")


    elapsed = time.time() - page_start

    print(
        f"\nPage processing time: "
        f"{elapsed:.2f} seconds"
    )


# ============================================================
# CLOSE PDF
# ============================================================

pdf.close()


# ============================================================
# SAVE TXT
# ============================================================

print("\n" + "=" * 80)
print("SAVING TXT")
print("=" * 80)

with open(
    TXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    for page in all_pages:

        f.write(
            f"\n{'=' * 80}\n"
        )

        f.write(
            f"الصفحة {page['page']}\n"
        )

        f.write(
            f"{'=' * 80}\n\n"
        )

        f.write(
            page["text"]
        )

        f.write("\n")


# ============================================================
# SAVE JSON
# ============================================================

print("\nSaving JSON...")

json_output = {
    "source_pdf": PDF_PATH,
    "pages": len(all_pages),
    "total_text_elements": total_text_lines,
    "pages_data": all_pages
}

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

total_time = time.time() - start_time

print("\n" + "=" * 80)
print("OCR COMPLETE")
print("=" * 80)

print(f"\nPages processed: {len(all_pages)}")
print(f"Total text elements: {total_text_lines}")

print(f"\nTXT saved to:")
print(TXT_PATH)

print(f"\nJSON saved to:")
print(JSON_PATH)

print(f"\nTotal processing time: {total_time / 60:.2f} minutes")


# ============================================================
# QUICK PREVIEW
# ============================================================

print("\n" + "=" * 80)
print("TEXT PREVIEW")
print("=" * 80)

preview_count = 0

for page in all_pages:

    if page["text"].strip():

        print(
            f"\n--- PAGE {page['page']} ---"
        )

        print(
            page["text"][:2000]
        )

        preview_count += 1

        if preview_count >= 3:
            break

if preview_count == 0:

    print(
        "\nWARNING: OCR did not return any recognized text."
    )

In [ ]:
import json
import re
import os

# ============================================================
# PATHS
# ============================================================

TXT_PATH = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output\arabic_ocr.txt"
JSON_PATH = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output\arabic_ocr.json"

OUTPUT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output"

CHUNKS_TXT = os.path.join(OUTPUT_DIR, "chunks.txt")
CHUNKS_JSON = os.path.join(OUTPUT_DIR, "chunks.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# LOAD FILES
# ============================================================

with open(TXT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

with open(JSON_PATH, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

print("TXT loaded")
print("JSON loaded")


# ============================================================
# NORMALIZATION
# Used ONLY for detecting structure.
# Original text is preserved.
# ============================================================

def normalize_arabic(text):

    text = text.replace("ـ", "")

    text = re.sub(r"[إأآٱ]", "ا", text)

    text = text.replace("ى", "ي")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# DOCUMENT TYPE
# ============================================================

def detect_document_type(text):

    t = normalize_arabic(text)

    if "قرار مجلس النواب" in t:
        return "قرار مجلس النواب"

    if "تعديل دستوري" in t:
        return "تعديل دستوري"

    if "قانون" in t:
        return "قانون"

    return None


# ============================================================
# ARTICLE DETECTION
# ============================================================

def is_article(text):

    t = normalize_arabic(text)

    # Normal OCR forms
    if re.fullmatch(r"المادة", t):
        return True

    if re.fullmatch(r"مادة", t):
        return True

    # OCR mistakes found in your document
    if re.fullmatch(r"الماة", t):
        return True

    if re.fullmatch(r"الادة", t):
        return True

    # مكرر
    if re.fullmatch(r"مادة\s*مكرر", t):
        return True

    if re.fullmatch(r"المادة\s*مكرر", t):
        return True

    if re.fullmatch(r"مادةمكرر", t):
        return True

    if re.fullmatch(r"المادةمكرر", t):
        return True

    return False


# ============================================================
# GET TEXT ELEMENTS FROM JSON
#
# JSON is used only to reconstruct the structure of the TXT.
# ============================================================

json_elements = []

for page in ocr_data["pages_data"]:

    page_number = page["page"]

    for element in page.get("elements", []):

        text = element.get("text", "").strip()

        if not text:
            continue

        json_elements.append({
            "page": page_number,
            "text": text,
            "confidence": element.get("confidence")
        })


# ============================================================
# BUILD TEXT FROM JSON
#
# This gives us a clean mapping:
#
# text element → page number
#
# But the actual chunking source remains the TXT.
# ============================================================

json_text_lines = []

for element in json_elements:
    json_text_lines.append(element["text"])


# ============================================================
# CHECK TXT / JSON SIMILARITY
# ============================================================

print()
print("TXT characters:", len(raw_text))
print("JSON elements :", len(json_elements))


# ============================================================
# CHUNK THE TXT
#
# We use article markers as the primary chunk boundaries.
# ============================================================

lines = [
    line.strip()
    for line in raw_text.splitlines()
    if line.strip()
]


chunks = []

current_lines = []

current_document_type = None
current_document_title = None
current_article = None

chunk_id = 1


# ============================================================
# PROCESS TXT LINE BY LINE
# ============================================================

for line in lines:

    normalized = normalize_arabic(line)

    # --------------------------------------------------------
    # Detect document type
    # --------------------------------------------------------

    detected_type = detect_document_type(line)

    if detected_type:

        # Only use it as metadata.
        # Do NOT automatically create a chunk.
        if current_document_type is None:

            current_document_type = detected_type


    # --------------------------------------------------------
    # Detect article
    # --------------------------------------------------------

    if is_article(line):

        # Save previous article
        if current_lines:

            chunk_text = "\n".join(current_lines).strip()

            if chunk_text:

                chunks.append({
                    "chunk_id": f"chunk_{chunk_id:04d}",
                    "document_type": current_document_type,
                    "document_title": current_document_title,
                    "article": current_article,
                    "text": chunk_text
                })

                chunk_id += 1

        # Start new article
        current_article = line

        current_lines = [line]

        continue


    # --------------------------------------------------------
    # Document title candidates
    # --------------------------------------------------------

    if (
        current_document_title is None
        and (
            "قرار مجلس النواب" in normalized
            or "قانون رقم" in normalized
            or "تعديل دستوري" in normalized
        )
    ):

        current_document_title = line


    # --------------------------------------------------------
    # Normal text
    # --------------------------------------------------------

    current_lines.append(line)


# ============================================================
# SAVE FINAL CHUNK
# ============================================================

if current_lines:

    chunk_text = "\n".join(current_lines).strip()

    if chunk_text:

        chunks.append({
            "chunk_id": f"chunk_{chunk_id:04d}",
            "document_type": current_document_type,
            "document_title": current_document_title,
            "article": current_article,
            "text": chunk_text
        })


# ============================================================
# MAP CHUNKS TO JSON PAGES
#
# We use the JSON to recover page information.
# ============================================================

for chunk in chunks:

    chunk_text = chunk["text"]

    normalized_chunk = normalize_arabic(chunk_text)

    matching_pages = []

    for element in json_elements:

        normalized_element = normalize_arabic(
            element["text"]
        )

        if not normalized_element:
            continue

        # Use distinctive OCR elements to associate
        # the chunk with its pages.
        if (
            len(normalized_element) >= 8
            and normalized_element in normalized_chunk
        ):

            matching_pages.append(element["page"])


    if matching_pages:

        chunk["page_start"] = min(matching_pages)
        chunk["page_end"] = max(matching_pages)

    else:

        chunk["page_start"] = None
        chunk["page_end"] = None


# ============================================================
# ADD EMBEDDING TEXT
#
# This is what will later go into the embedding model.
# ============================================================

for chunk in chunks:

    embedding_parts = []

    if chunk["document_type"]:
        embedding_parts.append(
            f"نوع الوثيقة: {chunk['document_type']}"
        )

    if chunk["document_title"]:
        embedding_parts.append(
            f"عنوان الوثيقة: {chunk['document_title']}"
        )

    if chunk["article"]:
        embedding_parts.append(
            f"المادة: {chunk['article']}"
        )

    embedding_parts.append(
        f"النص:\n{chunk['text']}"
    )

    chunk["embedding_text"] = "\n".join(
        embedding_parts
    )


# ============================================================
# SAVE CHUNKS JSON
# ============================================================

output = {
    "source_txt": TXT_PATH,
    "source_json": JSON_PATH,
    "chunking_method": "article_based",
    "total_chunks": len(chunks),
    "chunks": chunks
}

with open(CHUNKS_JSON, "w", encoding="utf-8") as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SAVE CHUNKS TXT
# ============================================================

with open(CHUNKS_TXT, "w", encoding="utf-8") as f:

    for chunk in chunks:

        f.write("=" * 80)
        f.write("\n")

        f.write(f"CHUNK: {chunk['chunk_id']}\n")
        f.write(
            f"TYPE: {chunk['document_type']}\n"
        )
        f.write(
            f"TITLE: {chunk['document_title']}\n"
        )
        f.write(
            f"ARTICLE: {chunk['article']}\n"
        )
        f.write(
            f"PAGES: {chunk['page_start']} - "
            f"{chunk['page_end']}\n"
        )

        f.write("-" * 80)
        f.write("\n")

        f.write(chunk["text"])
        f.write("\n\n")


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 60)
print("CHUNKING FINISHED")
print("=" * 60)

print("Total chunks:", len(chunks))

print()
print("Chunks TXT:")
print(CHUNKS_TXT)

print()
print("Chunks JSON:")
print(CHUNKS_JSON)

print()
print("First 3 chunks")
print("=" * 60)

for chunk in chunks[:3]:

    print()
    print("ID:", chunk["chunk_id"])
    print("Type:", chunk["document_type"])
    print("Article:", chunk["article"])
    print(
        "Pages:",
        chunk["page_start"],
        "-",
        chunk["page_end"]
    )

    print()
    print(chunk["text"][:500])
    print("...")

In [ ]:
import os
import json
import re

# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
OUTPUT_DIR = os.path.join(BASE_DIR, "ocr_output")

OCR_JSON = os.path.join(OUTPUT_DIR, "arabic_ocr.json")
OUTPUT_CHUNKS = os.path.join(OUTPUT_DIR, "chunks_v2.json")


# ============================================================
# LOAD OCR JSON
# ============================================================

with open(OCR_JSON, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

pages = ocr_data["pages_data"]

print("Pages loaded:", len(pages))


# ============================================================
# OCR NORMALIZATION
# Used ONLY for detecting structure.
# Original OCR text is preserved in the chunks.
# ============================================================

def normalize_for_detection(text):
    if not text:
        return ""

    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")
    text = text.replace("\ufeff", "")

    # Normalize common OCR variants
    text = text.replace("الماة", "المادة")
    text = text.replace("الادة", "المادة")
    text = text.replace("الماادة", "المادة")
    text = text.replace("مادةمكرر", "مادة مكرر")

    return text.strip()


# ============================================================
# ARTICLE DETECTION
# ============================================================

ARTICLE_PATTERNS = [
    r"^المادة\s*(الأولى|الثانية|الثالثة|الرابعة|الخامسة|السادسة|السابعة|الثامنة|التاسعة|العاشرة)$",
    r"^المادة\s*(الحادية عشرة|الثانية عشرة|الثالثة عشرة|الرابعة عشرة|الخامسة عشرة|السادسة عشرة|السابعة عشرة|الثامنة عشرة|التاسعة عشرة)$",
    r"^المادة\s*\S+",
    r"^مادة\s*$",
    r"^مادة\s+مكرر",
    r"^مادةمكرر",
    r"^المادة$"
]


def is_article_start(text):

    t = normalize_for_detection(text)

    if not t:
        return False

    for pattern in ARTICLE_PATTERNS:
        if re.match(pattern, t):
            return True

    return False


# ============================================================
# DOCUMENT START DETECTION
# ============================================================

def detect_document_start(text):

    t = normalize_for_detection(text)

    if not t:
        return False

    # Constitutional amendment
    if "تعديل دستوري" in t:
        return True

    # Parliament decisions
    if t.startswith("قرار مجلس النواب"):
        return True

    # Laws
    if t.startswith("قانون") or t.startswith("قانون رقم"):
        return True

    return False


# ============================================================
# DOCUMENT TYPE
# ============================================================

def detect_document_type(text):

    t = normalize_for_detection(text)

    if "تعديل دستوري" in t:
        return "تعديل دستوري"

    if t.startswith("قرار مجلس النواب"):
        return "قرار مجلس النواب"

    if t.startswith("قانون"):
        return "قانون"

    return None


# ============================================================
# CLEAN PAGE ARTIFACTS
# ============================================================

def clean_page_text(text):

    if not text:
        return ""

    lines = text.splitlines()

    cleaned = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Remove obvious page-layout artifacts
        if re.match(r"^=+$", line):
            continue

        if re.match(r"^الصفحة\s*\d+$", line):
            continue

        if re.match(r"^قم الصفحة", line):
            continue

        if line == "العدد":
            continue

        cleaned.append(line)

    return "\n".join(cleaned)


# ============================================================
# BUILD ORDERED OCR ELEMENTS
#
# Every element keeps its ORIGINAL page number.
# ============================================================

elements = []

for page in pages:

    page_number = page["page"]

    for element_index, element in enumerate(page.get("elements", [])):

        text = element.get("text", "").strip()

        if not text:
            continue

        elements.append({
            "page": page_number,
            "element_index": element_index,
            "text": text,
            "confidence": element.get("confidence"),
            "detection_confidence": element.get("detection_confidence"),
            "box": element.get("box")
        })


print("OCR elements:", len(elements))


# ============================================================
# DOCUMENT SEGMENTATION
#
# We detect actual document starts only.
# References such as:
#
# "على القانون رقم..."
#
# do NOT start a new document.
# ============================================================

documents = []

current_document = None

for element in elements:

    text = element["text"]
    normalized = normalize_for_detection(text)

    # --------------------------------------------------------
    # Detect document start
    # --------------------------------------------------------

    new_document = detect_document_start(normalized)

    if new_document:

        # Save previous document
        if current_document is not None:
            documents.append(current_document)

        current_document = {
            "document_type": detect_document_type(normalized),
            "title_parts": [text],
            "elements": [element],
            "page_start": element["page"],
            "page_end": element["page"]
        }

        continue

    # --------------------------------------------------------
    # If we haven't found a document yet
    # --------------------------------------------------------

    if current_document is None:
        continue

    # --------------------------------------------------------
    # Add element
    # --------------------------------------------------------

    current_document["elements"].append(element)

    current_document["page_end"] = element["page"]


# Save final document
if current_document is not None:
    documents.append(current_document)


print("Detected documents:", len(documents))


# ============================================================
# DOCUMENT TITLE EXTRACTION
# ============================================================

def extract_title(document):

    elements = document["elements"]

    title_parts = []

    for element in elements:

        text = element["text"]
        normalized = normalize_for_detection(text)

        # Stop title when legal body begins
        if "صدر القانون" in normalized:
            break

        if "صدر القرار" in normalized:
            break

        if "صدر التعديل" in normalized:
            break

        if is_article_start(normalized):
            break

        title_parts.append(text)

        # Prevent huge titles
        if len(" ".join(title_parts)) > 500:
            break

    title = " ".join(title_parts)

    title = re.sub(r"\s+", " ", title).strip()

    return title


# ============================================================
# CHUNK DOCUMENT INTO ARTICLES / PREAMBLE
# ============================================================

chunks = []

chunk_counter = 1


for document_index, document in enumerate(documents, start=1):

    document_id = f"document_{document_index:03d}"

    document_type = document["document_type"]

    document_title = extract_title(document)

    elements_doc = document["elements"]


    # --------------------------------------------------------
    # Find article boundaries
    # --------------------------------------------------------

    article_positions = []

    for i, element in enumerate(elements_doc):

        if is_article_start(element["text"]):
            article_positions.append(i)


    # --------------------------------------------------------
    # Build segments
    # --------------------------------------------------------

    segments = []

    if article_positions:

        # Preamble before first article
        if article_positions[0] > 0:

            preamble = elements_doc[
                0:article_positions[0]
            ]

            if preamble:
                segments.append({
                    "article": None,
                    "section_type": "preamble",
                    "elements": preamble
                })


        # Articles
        for j, start in enumerate(article_positions):

            if j + 1 < len(article_positions):
                end = article_positions[j + 1]
            else:
                end = len(elements_doc)

            article_elements = elements_doc[start:end]

            if not article_elements:
                continue

            article_name = normalize_for_detection(
                article_elements[0]["text"]
            )

            segments.append({
                "article": article_name,
                "section_type": "article",
                "elements": article_elements
            })

    else:

        # Document without detected articles
        segments.append({
            "article": None,
            "section_type": "document",
            "elements": elements_doc
        })


    # --------------------------------------------------------
    # Create chunks
    # --------------------------------------------------------

    for segment in segments:

        segment_elements = segment["elements"]

        raw_lines = []

        source_pages = []

        confidences = []

        for element in segment_elements:

            text = element["text"].strip()

            if not text:
                continue

            cleaned = clean_page_text(text)

            if not cleaned:
                continue

            raw_lines.append(cleaned)

            page = element["page"]

            if page not in source_pages:
                source_pages.append(page)

            confidence = element.get("confidence")

            if isinstance(confidence, (int, float)):
                confidences.append(float(confidence))


        if not raw_lines:
            continue


        # ----------------------------------------------------
        # Preserve original OCR text
        # ----------------------------------------------------

        chunk_text = "\n".join(raw_lines).strip()


        # ----------------------------------------------------
        # Correct page information
        # ----------------------------------------------------

        page_start = min(source_pages)
        page_end = max(source_pages)


        # ----------------------------------------------------
        # OCR statistics
        # ----------------------------------------------------

        if confidences:
            average_confidence = sum(confidences) / len(confidences)
            minimum_confidence = min(confidences)
        else:
            average_confidence = None
            minimum_confidence = None


        # ----------------------------------------------------
        # Embedding text
        # ----------------------------------------------------

        embedding_text = (
            f"نوع الوثيقة: {document_type}\n"
            f"عنوان الوثيقة: {document_title}\n"
            f"القسم: {segment['article'] or segment['section_type']}\n"
            f"النص:\n{chunk_text}"
        )


        chunk = {
            "chunk_id": f"chunk_{chunk_counter:04d}",
            "document_id": document_id,

            "document_type": document_type,
            "document_title": document_title,

            "section_type": segment["section_type"],
            "article": segment["article"],

            "page_start": page_start,
            "page_end": page_end,
            "source_pages": source_pages,

            "text": chunk_text,
            "embedding_text": embedding_text,

            "ocr": {
                "matched_elements": len(segment_elements),
                "average_confidence": average_confidence,
                "minimum_confidence": minimum_confidence
            }
        }

        chunks.append(chunk)

        chunk_counter += 1


# ============================================================
# SAVE
# ============================================================

output = {
    "source": OCR_JSON,
    "pages": len(pages),
    "documents": len(documents),
    "chunks": len(chunks),
    "chunks_data": chunks
}


with open(OUTPUT_CHUNKS, "w", encoding="utf-8") as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 70)
print("CHUNKING COMPLETE")
print("=" * 70)

print("Pages:", len(pages))
print("Documents:", len(documents))
print("Chunks:", len(chunks))

print()
print("Output:")
print(OUTPUT_CHUNKS)


# ============================================================
# DOCUMENT SUMMARY
# ============================================================

print()
print("=" * 70)
print("DOCUMENTS")
print("=" * 70)

for i, document in enumerate(documents, 1):

    print(
        f"{i:02d}. "
        f"{document['document_type']} | "
        f"pages {document['page_start']}-{document['page_end']} | "
        f"{extract_title(document)[:120]}"
    )


# ============================================================
# CHUNK SUMMARY
# ============================================================

print()
print("=" * 70)
print("FIRST 10 CHUNKS")
print("=" * 70)

for chunk in chunks[:10]:

    print()
    print(
        chunk["chunk_id"],
        "|",
        chunk["document_id"],
        "|",
        chunk["document_type"],
        "|",
        chunk["article"],
        "| pages",
        chunk["page_start"],
        "-",
        chunk["page_end"]
    )

    print(chunk["text"][:250].replace("\n", " "))

In [ ]:
import json
import os
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# PATHS
# ============================================================

CHUNKS_PATH = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output\chunks.json"

OUTPUT_PATH = r"C:\Users\PC-LAB1\Desktop\llm\ocr_output\embeddings.json"

# ============================================================
# SETTINGS
# ============================================================

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

BATCH_SIZE = 16

# ============================================================
# LOAD CHUNKS JSON
# ============================================================

print("Loading chunks.json...")

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Top-level type:", type(data).__name__)

# ============================================================
# FIND THE CHUNKS
# ============================================================

if isinstance(data, list):

    chunks = data

elif isinstance(data, dict):

    # Look for the chunks list
    if "chunks" in data and isinstance(data["chunks"], list):
        chunks = data["chunks"]

    elif "data" in data and isinstance(data["data"], list):
        chunks = data["data"]

    else:
        # Try to find a list containing chunk objects
        chunks = None

        for key, value in data.items():

            if isinstance(value, list) and len(value) > 0:

                if isinstance(value[0], dict) and "chunk_id" in value[0]:
                    chunks = value
                    print(f"Found chunks under key: {key}")
                    break

        if chunks is None:
            raise ValueError(
                "Could not find the chunks list inside chunks.json."
            )

else:
    raise ValueError("Unsupported chunks.json structure.")

# ============================================================
# CHECK CHUNKS
# ============================================================

print(f"Number of chunks: {len(chunks)}")

if len(chunks) == 0:
    raise ValueError("No chunks found.")

print("\nFirst chunk:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:2000])

# ============================================================
# EXTRACT EMBEDDING TEXT
# ============================================================

texts = []

for i, chunk in enumerate(chunks):

    text = chunk.get("embedding_text")

    if not text:
        text = chunk.get("text", "")

    if not isinstance(text, str):
        text = str(text)

    if not text.strip():
        print(f"Warning: chunk {i} has empty text.")

    texts.append(text)

print("\nTexts prepared:", len(texts))

# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model:")
print(MODEL_NAME)

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded.")

# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

print("\nGenerating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding generation complete.")

# ============================================================
# CHECK RESULTS
# ============================================================

print("Embedding matrix shape:", embeddings.shape)

print("Number of vectors:", embeddings.shape[0])

print("Vector dimensions:", embeddings.shape[1])

# ============================================================
# ADD EMBEDDINGS TO CHUNKS
# ============================================================

embedded_chunks = []

for chunk, embedding in zip(chunks, embeddings):

    item = dict(chunk)

    item["embedding"] = embedding.tolist()

    embedded_chunks.append(item)

# ============================================================
# SAVE EMBEDDINGS
# ============================================================

output_data = {
    "model": MODEL_NAME,
    "embedding_dimensions": int(embeddings.shape[1]),
    "normalized": True,
    "number_of_chunks": len(embedded_chunks),
    "chunks": embedded_chunks
}

os.makedirs(
    os.path.dirname(OUTPUT_PATH),
    exist_ok=True
)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# FILE SIZE
# ============================================================

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("EMBEDDING COMPLETE")
print("=" * 60)

print(f"Chunks:              {len(embedded_chunks)}")
print(f"Vector dimensions:   {embeddings.shape[1]}")
print(f"Output size:         {file_size_mb:.2f} MB")
print()
print("Saved to:")
print(OUTPUT_PATH)

print("=" * 60)

In [ ]:
import os
import json
import numpy as np
import faiss

# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
OUTPUT_DIR = os.path.join(BASE_DIR, "ocr_output")

EMBEDDINGS_FILE = os.path.join(OUTPUT_DIR, "embeddings.json")
VECTOR_DIR = os.path.join(OUTPUT_DIR, "vector_store")

os.makedirs(VECTOR_DIR, exist_ok=True)

INDEX_FILE = os.path.join(VECTOR_DIR, "index.faiss")
METADATA_FILE = os.path.join(VECTOR_DIR, "metadata.json")


# ============================================================
# LOAD EMBEDDINGS
# ============================================================

print("Loading embeddings...")

with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]

print("Number of chunks:", len(chunks))
print("Embedding dimensions:", data["embedding_dimensions"])


# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

embeddings = np.array(
    [chunk["embedding"] for chunk in chunks],
    dtype=np.float32
)

print("Embedding matrix shape:", embeddings.shape)


# ============================================================
# NORMALIZE
# ============================================================

# Your embeddings were created with normalize_embeddings=True,
# but normalizing again is safe.

faiss.normalize_L2(embeddings)


# ============================================================
# CREATE FAISS INDEX
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created.")
print("Vectors stored:", index.ntotal)


# ============================================================
# SAVE FAISS INDEX
# ============================================================

faiss.write_index(index, INDEX_FILE)

print("FAISS index saved:")
print(INDEX_FILE)


# ============================================================
# SAVE METADATA SEPARATELY
# ============================================================

metadata = []

for chunk in chunks:
    metadata.append({
        "chunk_id": chunk.get("chunk_id"),
        "document_id": chunk.get("document_id"),
        "document_type": chunk.get("document_type"),
        "document_title": chunk.get("document_title"),
        "section_type": chunk.get("section_type"),
        "article": chunk.get("article"),
        "page_start": chunk.get("page_start"),
        "page_end": chunk.get("page_end"),
        "source_pages": chunk.get("source_pages"),
        "text": chunk.get("text"),
        "embedding_text": chunk.get("embedding_text")
    })


with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Metadata saved:")
print(METADATA_FILE)

print("\nVector store complete.")

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
VECTOR_DIR = os.path.join(BASE_DIR, "ocr_output", "vector_store")

INDEX_FILE = os.path.join(VECTOR_DIR, "index.faiss")
METADATA_FILE = os.path.join(VECTOR_DIR, "metadata.json")


# ============================================================
# LOAD MODEL
# ============================================================

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# LOAD VECTOR STORE
# ============================================================

index = faiss.read_index(INDEX_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Vectors:", index.ntotal)
print("Metadata:", len(metadata))


# ============================================================
# SEARCH FUNCTION
# ============================================================

def search(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        chunk = metadata[idx]

        results.append({
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "document_type": chunk["document_type"],
            "document_title": chunk["document_title"],
            "article": chunk["article"],
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "text": chunk["text"]
        })

    return results


# ============================================================
# TEST QUERY
# ============================================================

query = "ما هي منحة أرباب الأسر؟"

results = search(query, top_k=5)


# ============================================================
# DISPLAY RESULTS
# ============================================================

for i, result in enumerate(results, 1):

    print("=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)

    print("Score:", round(result["score"], 4))
    print("Chunk:", result["chunk_id"])
    print("Document:", result["document_type"])
    print("Title:", result["document_title"])
    print("Article:", result["article"])
    print("Pages:", result["page_start"], "-", result["page_end"])

    print("\nText:")
    print(result["text"])

In [ ]:
import os
import json
import re
from collections import Counter


# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
OUTPUT_DIR = os.path.join(BASE_DIR, "ocr_output")

OCR_JSON = os.path.join(
    OUTPUT_DIR,
    "arabic_ocr.json"
)

OUTPUT_CHUNKS = os.path.join(
    OUTPUT_DIR,
    "chunks_final.json"
)


# ============================================================
# LOAD OCR JSON
# ============================================================

with open(OCR_JSON, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

pages = ocr_data.get("pages_data", [])

print("=" * 70)
print("LOADING OCR")
print("=" * 70)

print("Pages:", len(pages))


# ============================================================
# NORMALIZATION
#
# Used ONLY for structural detection.
# Original OCR text is never changed in the final chunk text.
# ============================================================

def normalize_detection(text):

    if not text:
        return ""

    text = str(text)

    # Invisible characters
    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")
    text = text.replace("\ufeff", "")

    # OCR variants
    replacements = {
        "الماة": "المادة",
        "الادة": "المادة",
        "الماادة": "المادة",

        "النية": "الثانية",
        "الثانيـة": "الثانية",
        "الثانيه": "الثانية",
        "الثاني": "الثانية",

        "مادةمكرر": "مادة مكرر",
        "مادةمكررة": "مادة مكرر"
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Normalize Arabic tatweel
    text = text.replace("ـ", "")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# CLEAN LAYOUT ARTIFACTS
#
# Only removes obvious OCR/page-layout noise.
# Legal text itself is preserved.
# ============================================================

def clean_text(text):

    if not text:
        return ""

    lines = text.splitlines()

    result = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Separator lines
        if re.fullmatch(r"=+", line):
            continue

        # Page headers/artifacts
        if re.fullmatch(r"الصفحة\s*\d+", line):
            continue

        if line.startswith("قم الصفحة"):
            continue

        result.append(line)

    return "\n".join(result).strip()


# ============================================================
# ARTICLE DETECTION
# ============================================================

ARTICLE_EXACT = {
    "مادة",
    "مادة مكرر",
    "المادة"
}


def is_article_start(text):

    t = normalize_detection(text)

    if not t:
        return False

    # Exact forms
    if t in ARTICLE_EXACT:
        return True

    # المادة الأولى
    # المادة الثانية
    # المادة الثالثة
    # المادة 1 ...
    if re.match(r"^المادة\s+.+$", t):
        return True

    # مادة مكرر
    if re.match(r"^مادة\s+مكرر", t):
        return True

    return False


# ============================================================
# DOCUMENT START DETECTION
#
# IMPORTANT:
# We only detect a document when the element itself starts
# with "قانون", "قرار مجلس النواب", or "تعديل دستوري".
#
# Therefore:
#
# "على القانون رقم..."
#
# will NOT start a new document.
# ============================================================

def detect_document_type(text):

    t = normalize_detection(text)

    if "تعديل دستوري" in t:
        return "تعديل دستوري"

    if t.startswith("قرار مجلس النواب"):
        return "قرار مجلس النواب"

    if t.startswith("قانون"):
        return "قانون"

    return None


def is_document_start(text):

    return detect_document_type(text) is not None


# ============================================================
# BUILD ORDERED OCR ELEMENTS
#
# Every OCR element retains its real page number.
# ============================================================

elements = []

for page in pages:

    page_number = page.get("page")

    for element_index, element in enumerate(
        page.get("elements", [])
    ):

        text = element.get("text", "").strip()

        if not text:
            continue

        elements.append({
            "page": page_number,
            "element_index": element_index,
            "text": text,
            "confidence": element.get("confidence"),
            "detection_confidence": element.get(
                "detection_confidence"
            ),
            "box": element.get("box")
        })


print("OCR elements:", len(elements))


# ============================================================
# DETECT DOCUMENTS
# ============================================================

documents = []

current_document = None


for element in elements:

    text = element["text"]

    if is_document_start(text):

        # Save previous document
        if current_document is not None:
            documents.append(current_document)

        document_type = detect_document_type(text)

        current_document = {
            "document_type": document_type,
            "elements": [],
            "page_start": element["page"],
            "page_end": element["page"]
        }

    # Ignore material before first document
    if current_document is None:
        continue

    current_document["elements"].append(element)

    current_document["page_end"] = element["page"]


# Save final document
if current_document is not None:
    documents.append(current_document)


print()
print("=" * 70)
print("DOCUMENT DETECTION")
print("=" * 70)

print("Documents detected:", len(documents))

for i, document in enumerate(documents, 1):

    print(
        f"{i:02d}. "
        f"{document['document_type']} | "
        f"pages {document['page_start']}-"
        f"{document['page_end']}"
    )


# ============================================================
# TITLE EXTRACTION
# ============================================================

def extract_document_title(document):

    title_parts = []

    for element in document["elements"]:

        text = element["text"]
        normalized = normalize_detection(text)

        # Stop at legal operative section
        if "صدر القانون" in normalized:
            break

        if "صدر القرار" in normalized:
            break

        if "صدر التعديل" in normalized:
            break

        if is_article_start(text):
            break

        # Don't let obvious closing material become title
        if normalized in {
            "مجلس النواب",
            "الحكومة",
            "رئيس مجلس النواب"
        }:
            break

        title_parts.append(text)

        # Safety limit
        if len(" ".join(title_parts)) > 600:
            break

    title = " ".join(title_parts)

    title = re.sub(r"\s+", " ", title).strip()

    return title


# ============================================================
# SPLIT DOCUMENT INTO PREAMBLE + ARTICLES
# ============================================================

all_chunks = []

chunk_counter = 1


for document_number, document in enumerate(
    documents,
    start=1
):

    document_id = f"document_{document_number:03d}"

    document_type = document["document_type"]

    document_title = extract_document_title(
        document
    )

    document_elements = document["elements"]


    # --------------------------------------------------------
    # Locate article starts
    # --------------------------------------------------------

    article_positions = []

    for i, element in enumerate(document_elements):

        if is_article_start(element["text"]):
            article_positions.append(i)


    # --------------------------------------------------------
    # Build sections
    # --------------------------------------------------------

    sections = []


    # --------------------------------------------------------
    # PREAMBLE
    # --------------------------------------------------------

    if article_positions:

        first_article = article_positions[0]

        if first_article > 0:

            sections.append({
                "section_type": "preamble",
                "article": None,
                "elements": document_elements[
                    :first_article
                ]
            })


        # ----------------------------------------------------
        # ARTICLES
        # ----------------------------------------------------

        for i, start in enumerate(article_positions):

            if i + 1 < len(article_positions):

                end = article_positions[i + 1]

            else:

                end = len(document_elements)


            article_elements = document_elements[
                start:end
            ]

            if not article_elements:
                continue


            article_name = normalize_detection(
                article_elements[0]["text"]
            )


            sections.append({
                "section_type": "article",
                "article": article_name,
                "elements": article_elements
            })


    else:

        # Document without detectable article
        sections.append({
            "section_type": "document",
            "article": None,
            "elements": document_elements
        })


    # --------------------------------------------------------
    # CREATE CHUNKS
    # --------------------------------------------------------

    for section in sections:

        section_elements = section["elements"]

        text_parts = []

        source_pages = []

        confidences = []

        matched_elements = 0


        for element in section_elements:

            raw_text = element["text"]

            cleaned = clean_text(raw_text)

            if not cleaned:
                continue


            text_parts.append(cleaned)

            matched_elements += 1


            # REAL PAGE NUMBER
            page_number = element["page"]

            if page_number not in source_pages:
                source_pages.append(page_number)


            # OCR confidence
            confidence = element.get("confidence")

            if isinstance(
                confidence,
                (int, float)
            ):
                confidences.append(
                    float(confidence)
                )


        if not text_parts:
            continue


        # ----------------------------------------------------
        # ORIGINAL OCR TEXT
        # ----------------------------------------------------

        chunk_text = "\n".join(
            text_parts
        ).strip()


        # ----------------------------------------------------
        # PAGE INFORMATION
        # ----------------------------------------------------

        page_start = min(source_pages)

        page_end = max(source_pages)


        # ----------------------------------------------------
        # CONFIDENCE
        # ----------------------------------------------------

        if confidences:

            average_confidence = (
                sum(confidences)
                / len(confidences)
            )

            minimum_confidence = min(
                confidences
            )

        else:

            average_confidence = None
            minimum_confidence = None


        # ----------------------------------------------------
        # EMBEDDING TEXT
        # ----------------------------------------------------

        embedding_text = (
            f"نوع الوثيقة: {document_type}\n"
            f"عنوان الوثيقة: {document_title}\n"
            f"القسم: "
            f"{section['article'] or section['section_type']}\n"
            f"النص:\n"
            f"{chunk_text}"
        )


        # ----------------------------------------------------
        # CHUNK
        # ----------------------------------------------------

        chunk = {

            "chunk_id":
                f"chunk_{chunk_counter:04d}",

            "document_id":
                document_id,

            "document_type":
                document_type,

            "document_title":
                document_title,

            "section_type":
                section["section_type"],

            "article":
                section["article"],

            "page_start":
                page_start,

            "page_end":
                page_end,

            "source_pages":
                source_pages,

            "text":
                chunk_text,

            "embedding_text":
                embedding_text,

            "ocr": {

                "matched_elements":
                    matched_elements,

                "average_confidence":
                    average_confidence,

                "minimum_confidence":
                    minimum_confidence
            }
        }


        all_chunks.append(chunk)

        chunk_counter += 1


# ============================================================
# OUTPUT
# ============================================================

output = {

    "source_pdf":
        ocr_data.get("source_pdf"),

    "source_ocr_json":
        OCR_JSON,

    "pages":
        len(pages),

    "documents":
        len(documents),

    "total_chunks":
        len(all_chunks),

    "chunks":
        all_chunks
}


with open(
    OUTPUT_CHUNKS,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("FINAL CHUNKING COMPLETE")
print("=" * 70)

print("Pages:", len(pages))
print("Documents:", len(documents))
print("Chunks:", len(all_chunks))

print()
print("Saved to:")
print(OUTPUT_CHUNKS)


# ============================================================
# DOCUMENT SUMMARY
# ============================================================

print()
print("=" * 70)
print("DOCUMENT SUMMARY")
print("=" * 70)

for document_number, document in enumerate(
    documents,
    start=1
):

    document_id = (
        f"document_{document_number:03d}"
    )

    document_chunks = [
        c
        for c in all_chunks
        if c["document_id"] == document_id
    ]

    print(
        f"{document_id} | "
        f"{document['document_type']} | "
        f"pages "
        f"{document['page_start']}-"
        f"{document['page_end']} | "
        f"{len(document_chunks)} chunks"
    )

    title = extract_document_title(
        document
    )

    print(
        "  Title:",
        title[:150]
    )


# ============================================================
# CHUNK SUMMARY
# ============================================================

print()
print("=" * 70)
print("ALL CHUNKS")
print("=" * 70)

for chunk in all_chunks:

    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['document_id']} | "
        f"{chunk['document_type']} | "
        f"{chunk['article']} | "
        f"pages "
        f"{chunk['page_start']}-"
        f"{chunk['page_end']}"
    )

    preview = (
        chunk["text"]
        .replace("\n", " ")
    )

    print(
        "   ",
        preview[:180]
    )


# ============================================================
# PAGE COVERAGE CHECK
# ============================================================

print()
print("=" * 70)
print("PAGE COVERAGE")
print("=" * 70)

page_chunk_counts = Counter()

for chunk in all_chunks:

    for page in chunk["source_pages"]:

        page_chunk_counts[page] += 1


for page in range(1, len(pages) + 1):

    print(
        f"Page {page:02d}: "
        f"{page_chunk_counts.get(page, 0)} chunk(s)"
    )

In [ ]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
OUTPUT_DIR = os.path.join(BASE_DIR, "ocr_output")

CHUNKS_FILE = os.path.join(
    OUTPUT_DIR,
    "chunks_final.json"
)

EMBEDDINGS_FILE = os.path.join(
    OUTPUT_DIR,
    "embeddings_final.json"
)


# ============================================================
# LOAD CHUNKS
# ============================================================

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]

print("Chunks loaded:", len(chunks))


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)


# ============================================================
# PREPARE TEXT
# ============================================================

texts = [
    chunk["embedding_text"]
    for chunk in chunks
]


# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

print("Generating embeddings...")

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print("Embedding shape:", embeddings.shape)


# ============================================================
# ADD EMBEDDINGS TO CHUNKS
# ============================================================

for chunk, embedding in zip(
    chunks,
    embeddings
):

    chunk["embedding"] = embedding.tolist()


# ============================================================
# SAVE
# ============================================================

output = {
    "model": MODEL_NAME,
    "embedding_dimensions": int(
        embeddings.shape[1]
    ),
    "normalized": True,
    "number_of_chunks": len(chunks),
    "chunks": chunks
}


with open(
    EMBEDDINGS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False
    )


print()
print("=" * 60)
print("EMBEDDINGS COMPLETE")
print("=" * 60)

print("Chunks:", len(chunks))
print("Dimensions:", embeddings.shape[1])
print("Saved:", EMBEDDINGS_FILE)

In [ ]:
import os
import json
import faiss
import numpy as np

# ============================================================
# Paths
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"

EMBEDDINGS_FILE = os.path.join(
    BASE_DIR,
    "ocr_output",
    "embeddings_final.json"
)

VECTOR_DIR = os.path.join(
    BASE_DIR,
    "vector_store_final"
)

os.makedirs(VECTOR_DIR, exist_ok=True)

INDEX_FILE = os.path.join(VECTOR_DIR, "index.faiss")
METADATA_FILE = os.path.join(VECTOR_DIR, "metadata.json")


# ============================================================
# Load embeddings
# ============================================================

with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]

print("Chunks:", len(chunks))


# ============================================================
# Get embeddings
# ============================================================

embeddings = np.array(
    [chunk["embedding"] for chunk in chunks],
    dtype=np.float32
)

print("Embedding matrix:", embeddings.shape)


# ============================================================
# Detect embedding dimensions automatically
# ============================================================

dimension = embeddings.shape[1]

print("Embedding dimensions:", dimension)


# ============================================================
# Create FAISS index
# ============================================================

# Embeddings were normalized during embedding generation.
# Therefore Inner Product gives cosine similarity.

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)


# ============================================================
# Save FAISS index
# ============================================================

faiss.write_index(index, INDEX_FILE)


# ============================================================
# Create metadata
# ============================================================

metadata = []

for chunk in chunks:

    metadata.append({
        "chunk_id": chunk.get("chunk_id"),
        "document_id": chunk.get("document_id"),
        "document_type": chunk.get("document_type"),
        "document_title": chunk.get("document_title"),
        "section_type": chunk.get("section_type"),
        "article": chunk.get("article"),
        "page_start": chunk.get("page_start"),
        "page_end": chunk.get("page_end"),
        "source_pages": chunk.get("source_pages"),
        "text": chunk.get("text")
    })


# ============================================================
# Save metadata
# ============================================================

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Verify
# ============================================================

print("\n" + "=" * 60)
print("FAISS VECTOR STORE CREATED")
print("=" * 60)

print("Vectors:", index.ntotal)
print("Dimensions:", index.d)
print("Index:", INDEX_FILE)
print("Metadata:", METADATA_FILE)

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# Paths
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"

VECTOR_DIR = os.path.join(
    BASE_DIR,
    "vector_store_final"
)

INDEX_FILE = os.path.join(
    VECTOR_DIR,
    "index.faiss"
)

METADATA_FILE = os.path.join(
    VECTOR_DIR,
    "metadata.json"
)


# ============================================================
# Load FAISS
# ============================================================

index = faiss.read_index(INDEX_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata:", len(metadata))


# ============================================================
# Load embedding model
# ============================================================

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# Retrieval function
# ============================================================

def search_legal_documents(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        item = metadata[idx]

        results.append({
            "score": float(score),
            "chunk_id": item.get("chunk_id"),
            "document_id": item.get("document_id"),
            "document_type": item.get("document_type"),
            "document_title": item.get("document_title"),
            "section_type": item.get("section_type"),
            "article": item.get("article"),
            "page_start": item.get("page_start"),
            "page_end": item.get("page_end"),
            "text": item.get("text")
        })

    return results


# ============================================================
# TEST QUERY
# ============================================================

query = "أرباب الأسر"

results = search_legal_documents(
    query,
    top_k=5
)


# ============================================================
# Display results
# ============================================================

print("\n")
print("=" * 80)
print("QUERY")
print("=" * 80)

print(query)

print("\n")
print("=" * 80)
print("RETRIEVED RESULTS")
print("=" * 80)


for i, result in enumerate(results, 1):

    print(f"\n{'=' * 80}")
    print(f"RESULT {i}")
    print(f"{'=' * 80}")

    print("Score:", round(result["score"], 4))
    print("Chunk:", result["chunk_id"])
    print("Document:", result["document_id"])
    print("Type:", result["document_type"])
    print("Title:", result["document_title"])
    print("Section:", result["section_type"])
    print("Article:", result["article"])
    print(
        "Pages:",
        result["page_start"],
        "→",
        result["page_end"]
    )

    print("\nTEXT:")
    print(result["text"])

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# Paths
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"

VECTOR_DIR = os.path.join(
    BASE_DIR,
    "vector_store_final"
)

INDEX_FILE = os.path.join(
    VECTOR_DIR,
    "index.faiss"
)

METADATA_FILE = os.path.join(
    VECTOR_DIR,
    "metadata.json"
)


# ============================================================
# Load FAISS vector store
# ============================================================

index = faiss.read_index(INDEX_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata entries:", len(metadata))


# ============================================================
# Load embedding model
# ============================================================

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embedding model loaded.")


# ============================================================
# Retrieval function
# ============================================================

def retrieve_context(query, top_k=5):

    # --------------------------------------------------------
    # Convert question to embedding
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Search FAISS
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0:
            continue

        item = metadata[idx]

        results.append({
            "score": float(score),
            "chunk_id": item.get("chunk_id"),
            "document_id": item.get("document_id"),
            "document_type": item.get("document_type"),
            "document_title": item.get("document_title"),
            "section_type": item.get("section_type"),
            "article": item.get("article"),
            "page_start": item.get("page_start"),
            "page_end": item.get("page_end"),
            "source_pages": item.get("source_pages"),
            "text": item.get("text", "")
        })

    return results


# ============================================================
# Build RAG context
# ============================================================

def build_rag_context(results):

    context_parts = []

    for i, result in enumerate(results, 1):

        source = f"""
[المصدر {i}]
معرّف المقطع: {result["chunk_id"]}
نوع الوثيقة: {result["document_type"]}
عنوان الوثيقة: {result["document_title"]}
القسم: {result["section_type"]}
المادة: {result["article"]}
الصفحات: {result["page_start"]} - {result["page_end"]}

النص:
{result["text"]}
"""

        context_parts.append(source.strip())

    return "\n\n" + "\n\n".join(context_parts)


# ============================================================
# Ask a question
# ============================================================

query = "أرباب الأسر؟"

results = retrieve_context(
    query,
    top_k=5
)

context = build_rag_context(results)


# ============================================================
# Display retrieval results
# ============================================================

print("\n")
print("=" * 80)
print("USER QUESTION")
print("=" * 80)

print(query)


print("\n")
print("=" * 80)
print("RETRIEVED CHUNKS")
print("=" * 80)

for i, result in enumerate(results, 1):

    print(f"\n--- Result {i} ---")

    print(
        "Similarity:",
        round(result["score"], 4)
    )

    print(
        "Chunk:",
        result["chunk_id"]
    )

    print(
        "Document:",
        result["document_id"]
    )

    print(
        "Type:",
        result["document_type"]
    )

    print(
        "Article:",
        result["article"]
    )

    print(
        "Pages:",
        result["page_start"],
        "→",
        result["page_end"]
    )

    print("\nText:")
    print(result["text"])


# ============================================================
# Display final RAG context
# ============================================================

print("\n")
print("=" * 80)
print("RAG CONTEXT")
print("=" * 80)

print(context)

In [ ]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ============================================================
# PATHS
# ============================================================

BASE_DIR = r"C:\Users\PC-LAB1\Desktop\llm"
VECTOR_DIR = os.path.join(BASE_DIR, "vector_store_final")

INDEX_FILE = os.path.join(VECTOR_DIR, "index.faiss")
METADATA_FILE = os.path.join(VECTOR_DIR, "metadata.json")

# ============================================================
# LOAD FAISS
# ============================================================

index = faiss.read_index(INDEX_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata entries:", len(metadata))

# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ============================================================
# RETRIEVAL FUNCTION
# ============================================================

def retrieve(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):

        if idx < 0:
            continue

        item = metadata[idx]

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": item.get("chunk_id"),
            "document_id": item.get("document_id"),
            "document_type": item.get("document_type"),
            "document_title": item.get("document_title"),
            "section_type": item.get("section_type"),
            "article": item.get("article"),
            "page_start": item.get("page_start"),
            "page_end": item.get("page_end"),
            "source_pages": item.get("source_pages"),
            "text": item.get("text", "")
        })

    return results


# ============================================================
# DISPLAY RESULTS
# ============================================================

def test_query(query, top_k=5):

    print("=" * 100)
    print("QUERY:")
    print(query)
    print("=" * 100)

    results = retrieve(query, top_k)

    for result in results:

        print(f"\n#{result['rank']}  Similarity: {result['score']:.4f}")
        print(f"Chunk: {result['chunk_id']}")
        print(f"Document: {result['document_id']}")
        print(f"Type: {result['document_type']}")
        print(f"Article: {result['article']}")
        print(
            f"Pages: "
            f"{result['page_start']} - {result['page_end']}"
        )

        print("-" * 80)
        print(result["text"])


# ============================================================
# TEST QUESTIONS
# ============================================================

test_questions = [
    "ما هي أحكام أرباب الأسر؟",
    "ما هي المادة المتعلقة بسحب الثقة عن الحكومة؟",
    "ما هي أحكام القانون؟",
    "من يتولى رئاسة المحكمة العليا؟",
    "متى يعمل بأحكام القانون؟"
]

for question in test_questions:
    test_query(question, top_k=5)
    print("\n\n")